# LST Regression Model + Scenario Simulation

Trains a model to **predict Land Surface Temperature (LST)** from urban morphology features.
The trained model is then used to simulate the LST reduction that green walls would cause by increasing local NDVI.

| | |
|---|---|
| **Target (y)** | `mean_LST` per 100×100m cell |
| **Features (X)** | NDVI, pct_green, pct_built, building height, building density, dist to road, dist to water |
| **Models** | Linear Regression → Random Forest → XGBoost |
| **Comparison** | Full features (with NDVI) vs Urban morphology only (no NDVI) |
| **Scenario** | Increase NDVI by green-wall delta → re-predict LST → compute cooling impact |

### Data quality note
The GEE-exported LST raster has no nodata tag set — sea/masked pixels were stored as `0` instead of `NaN`.
This causes ~354 coastal cells to have `mean_LST=0`, which are invalid and filtered out before training (`LST > 5°C`).

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import xgboost as xgb

OUT = Path('../outputs')
OUT.mkdir(exist_ok=True)

FEATURES = ['mean_NDVI','pct_green','pct_built','mean_building_height',
            'building_density','dist_major_road','dist_to_water']
FEATURES_MORPH = ['pct_green','pct_built','mean_building_height',
                  'building_density','dist_major_road','dist_to_water']
TARGET = 'mean_LST'
print('Setup OK')

## 1 — Load & Clean Data

In [ ]:
grid_raw = gpd.read_file('../outputs/tel_aviv_grid_features.gpkg')
print(f'Total cells   : {len(grid_raw):,}')
print(f'LST=0 cells   : {(grid_raw["mean_LST"]==0).sum()}  ← sea/nodata, filtered out')

# Filter: LST=0 are invalid coastal cells where GEE stored masked pixels as 0
grid = grid_raw[grid_raw['mean_LST'] > 5].dropna(subset=FEATURES+[TARGET]).reset_index(drop=True)
print(f'Valid cells   : {len(grid):,}  (LST {grid[TARGET].min():.1f}–{grid[TARGET].max():.1f}°C)')
grid[FEATURES + [TARGET]].describe().round(2)

## 2 — Train / Test Split

In [ ]:
X  = grid[FEATURES].values
Xm = grid[FEATURES_MORPH].values
y  = grid[TARGET].values

X_tr, X_te, Xm_tr, Xm_te, y_tr, y_te = train_test_split(
    X, Xm, y, test_size=0.2, random_state=42
)
print(f'Train: {len(X_tr):,}  |  Test: {len(X_te):,}')

## 3 — Train Models (full features + morphology only)

In [ ]:
def make_models():
    return {
        'Linear Regression': Pipeline([('sc', StandardScaler()), ('m', LinearRegression())]),
        'Random Forest'    : RandomForestRegressor(n_estimators=300, max_depth=12, min_samples_leaf=5, n_jobs=-1, random_state=42),
        'XGBoost'          : xgb.XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05,
                                               subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0),
    }

def train_eval(models, X_tr, X_te, y_tr, y_te, label):
    results = {}
    print(f'\n=== {label} ===')
    for name, model in models.items():
        model.fit(X_tr, y_tr)
        p = model.predict(X_te)
        results[name] = {'MAE': mean_absolute_error(y_te,p),
                         'RMSE': root_mean_squared_error(y_te,p),
                         'R2': r2_score(y_te,p), 'preds': p}
        print(f'  {name:20s}  R²={results[name]["R2"]:.3f}  MAE={results[name]["MAE"]:.2f}°C')
    return results

models_full  = make_models()
models_morph = make_models()
results_full  = train_eval(models_full,  X_tr,  X_te,  y_tr, y_te, 'Full features (with NDVI)')
results_morph = train_eval(models_morph, Xm_tr, Xm_te, y_tr, y_te, 'Urban morphology only (no NDVI)')

best_name  = max(results_full,  key=lambda k: results_full[k]['R2'])
best_model = models_full[best_name]
print(f'\nBest model: {best_name}')

## 4 — Diagnostics: Actual vs Predicted + Model Comparison + Residuals

In [ ]:
grid['predicted_LST'] = best_model.predict(X)
grid['residual']      = grid[TARGET] - grid['predicted_LST']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Actual vs predicted
ax = axes[0]
bp = results_full[best_name]['preds']
ax.scatter(y_te, bp, alpha=0.4, s=10, color='steelblue')
lims = [y_te.min()-0.5, y_te.max()+0.5]
ax.plot(lims, lims, 'r--', lw=1.5)
ax.set_xlabel('Actual LST (°C)'); ax.set_ylabel('Predicted LST (°C)')
ax.set_title(f'{best_name}\nR²={results_full[best_name]["R2"]:.3f}  MAE={results_full[best_name]["MAE"]:.2f}°C', fontweight='bold')
ax.set_xlim(lims); ax.set_ylim(lims)

# Model comparison bar chart
ax = axes[1]
model_names = list(results_full.keys())
r2_f = [results_full[n]['R2']  for n in model_names]
r2_m = [results_morph[n]['R2'] for n in model_names]
x = np.arange(len(model_names)); w = 0.35
b1 = ax.bar(x-w/2, r2_f, w, label='Full (with NDVI)',  color='steelblue')
b2 = ax.bar(x+w/2, r2_m, w, label='Morphology only',   color='coral')
ax.set_xticks(x); ax.set_xticklabels([n.replace(' ','\n') for n in model_names], fontsize=9)
ax.set_ylabel('R²'); ax.set_title('Model Comparison', fontweight='bold')
ax.legend(); ax.set_ylim(0, 0.55)
for bar in list(b1)+list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)

# Residuals map
grid_wgs = grid.to_crs(4326)
grid_wgs.plot(column='residual', cmap='coolwarm', legend=True,
              legend_kwds={'label':'Residual (°C)', 'shrink':0.7}, ax=axes[2])
axes[2].set_title('Residuals (Actual − Predicted)', fontweight='bold'); axes[2].axis('off')

plt.tight_layout()
plt.savefig(OUT / 'model_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

## 5 — Feature Importance

Two models side by side reveal different stories:
- **Full model:** NDVI is the strongest single predictor (~35%) — vegetation is the dominant cooling mechanism.
- **Morphology model:** without NDVI, green coverage (`pct_green`), distance to water, and road proximity become the main drivers — this reflects the urban structure contribution to heat independently of vegetation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

rf_f = models_full['Random Forest']
imp_f = rf_f.feature_importances_
order_f = np.argsort(imp_f)
axes[0].barh([FEATURES[i] for i in order_f], imp_f[order_f], color='steelblue')
axes[0].set_title('Random Forest\nFull features (with NDVI)', fontweight='bold')
axes[0].set_xlabel('Importance')

rf_m = models_morph['Random Forest']
imp_m = rf_m.feature_importances_
order_m = np.argsort(imp_m)
axes[1].barh([FEATURES_MORPH[i] for i in order_m], imp_m[order_m], color='coral')
axes[1].set_title('Random Forest\nUrban morphology only (no NDVI)', fontweight='bold')
axes[1].set_xlabel('Importance')

plt.suptitle('Feature Importance', fontweight='bold')
plt.tight_layout()
plt.savefig(OUT / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 6 — Scenario Simulation: Green Wall Impact

For each cell, estimate the NDVI increase that installing green walls on surrounding buildings would cause, then re-predict LST.

In [ ]:
COVERAGE_RATIO = 0.4
CELL_AREA = 100 * 100

avg_footprint = (grid['pct_built'] * CELL_AREA / grid['building_density'].replace(0, np.nan)).fillna(0)
avg_perimeter = 4 * np.sqrt(avg_footprint.clip(1))
wall_area_per_cell = avg_perimeter * grid['mean_building_height'] * COVERAGE_RATIO * grid['building_density']
ndvi_delta = (wall_area_per_cell / CELL_AREA).clip(0, 0.25)

X_scenario = grid[FEATURES].copy()
X_scenario['mean_NDVI'] = (X_scenario['mean_NDVI'] + ndvi_delta).clip(upper=1.0)
X_scenario['pct_green'] = (X_scenario['pct_green'] + ndvi_delta * 0.5).clip(upper=1.0)

grid['predicted_LST_with_walls'] = best_model.predict(X_scenario.values)
grid['predicted_cooling']        = (grid['predicted_LST'] - grid['predicted_LST_with_walls']).clip(lower=0)

print(f'Mean predicted cooling : {grid["predicted_cooling"].mean():.2f}°C')
print(f'Max predicted cooling  : {grid["predicted_cooling"].max():.2f}°C')
print(f'Cells with >0.5°C gain : {(grid["predicted_cooling"] > 0.5).sum():,}')

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
grid_wgs = grid.to_crs(4326)
grid_wgs.plot(column='predicted_LST', cmap='RdYlBu_r', legend=True,
              legend_kwds={'label':'Predicted LST (°C)', 'shrink':0.7}, ax=axes[0])
axes[0].set_title('Predicted LST (baseline)', fontweight='bold'); axes[0].axis('off')
grid_wgs.plot(column='predicted_cooling', cmap='Blues', legend=True,
              legend_kwds={'label':'Cooling (°C)', 'shrink':0.7}, ax=axes[1])
axes[1].set_title('Predicted Cooling from Green Walls', fontweight='bold'); axes[1].axis('off')
plt.tight_layout()
plt.savefig(OUT / 'scenario_simulation.png', dpi=150, bbox_inches='tight')
plt.show()

## 7 — Save

In [ ]:
joblib.dump(best_model, OUT / 'lst_model.pkl')
grid.to_file(OUT / 'tel_aviv_grid_features.gpkg', driver='GPKG')
print(f'Model : outputs/lst_model.pkl  ({best_name})')
print(f'Grid  : outputs/tel_aviv_grid_features.gpkg')
print(f'Columns added: predicted_LST, residual, predicted_LST_with_walls, predicted_cooling')